# Etapa 5c — Umbral con histéresis y limpieza por forma (prueba de mejora 1)

**Pregunta.** El algoritmo de la etapa 5 detecta el 11 % de los vasos finos (línea central). ¿Un umbral con histéresis, que acepta píxeles débiles conectados a vasos seguros, y una limpieza que quita manchas no alargadas mejoran la segmentación, sobre todo en los vasos finos?

**Mapa.** El de la etapa 5 a la mitad de resolución: verde → corrección del fondo (gaussiana, σ = 0,02 del diámetro) → detector de líneas, agrandado al tamaño original.

**Candidatas de umbral** (`src/segment.py`):
- Otsu en cada imagen (la referencia de la etapa 5);
- fracción fija de píxeles del campo: 6 % a 16 %;
- histéresis (`segmentar_histeresis`): fracción alta 4 %, 6 %, 8 % o 10 %, y fracción baja entre 2 y 10 puntos más.

**Limpieza por forma** (`limpiar_por_forma`), sobre Otsu y sobre cada histéresis: alargamiento mínimo 2, 4 u 8; área máxima de lo que se puede quitar, 0,01 % o 0,05 % del campo.

**Cómo se elige** (validación cruzada por personas, 5 grupos): en cada vuelta, con las personas de entrenamiento, la regla de umbral con mayor Matthews medio; después, encima de ella, la limpieza con mayor Matthews medio entre las que no bajan más de 0,02 la sensibilidad de línea central en vasos finos (esquema, etapa 5). Se mide en las personas de afuera, a resolución original y con todos los píxeles.

**Cuándo se adopta** (fijado antes de correr): la histéresis o la limpieza se adoptan si (a) no son inferiores a la referencia en Matthews (margen −0,03, desvío 14.4) y (b) mejoran la sensibilidad de línea central en vasos finos con un intervalo al 95 % que no incluye el cero. Si solo cumplen (a), gana la más simple (Otsu sin limpieza).


In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
FACTOR = 2
FRACS = [0.06, 0.07, 0.08, 0.09, 0.10, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16]
HYST = [(a, round(a + d, 2)) for a in (0.04, 0.06, 0.08, 0.10) for d in (0.02, 0.04, 0.06, 0.08, 0.10)]
CLEAN = [(r, s) for r in (2, 4, 8) for s in (1e-4, 5e-4)]

## Una pasada: todas las máscaras de cada imagen

In [2]:
def run(image_id):
    import numpy as np
    from scipy import ndimage as ndi
    from skimage.morphology import skeletonize
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import apply_filter, prepare_maps, upscale
    from src.fov import detect_fov_robust
    from src.metrics import matthews
    from src.preprocess import corregir_fondo
    from src.segment import segmentar_fraccion, segmentar_histeresis, segmentar_otsu
    from src.vessels import skeleton_widths

    rgb = load_image(image_id)
    ref, fov = load_vessels(image_id), load_fov(image_id)
    maps, region, d = prepare_maps(rgb, ["Verde"], FACTOR)
    m = corregir_fondo(maps["Verde"], region, 0.02 * d)
    score = upscale(apply_filter("detector de líneas", m, d), fov.shape)
    auto = detect_fov_robust(rgb)
    skel, w = skeleton_widths(ref & fov)
    thin = skel & (w <= WIDTH_LIMITS_PX[0])
    fov_area = auto.sum()
    rows = []

    def record(rule, clean, mask):
        rows.append({"image": image_id, "regla": rule, "limpieza": clean,
                     "matthews": matthews(mask, ref, fov), "finos": mask[thin].mean()})

    def with_cleaning(rule, mask):
        record(rule, "ninguna", mask)
        lab, n = ndi.label(mask, structure=np.ones((3, 3), dtype=int))
        area = np.bincount(lab.ravel(), minlength=n + 1)
        length = np.bincount(lab.ravel(), weights=skeletonize(mask).ravel(), minlength=n + 1)
        elong = length ** 2 / np.maximum(area, 1)
        for r, s in CLEAN:
            drop = (area <= s * fov_area) & (elong < r)
            drop[0] = False
            record(rule, f"alargamiento {r}, área {s:g}", mask & ~drop[lab])

    with_cleaning("Otsu", segmentar_otsu(score, auto)[0])
    for f in FRACS:
        record(f"fracción {f:.2f}", "ninguna", segmentar_fraccion(score, auto, f)[0])
    for a, b in HYST:
        with_cleaning(f"histéresis {a:.2f}-{b:.2f}", segmentar_histeresis(score, auto, a, b)[0])
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(run, ids, workers=4) for r in rows])
R.to_csv(ROOT / "results" / "05c_histeresis_forma.csv", index=False)
print(f"{time.time() - t:.0f} s; {len(R)} filas")

554 s; 4740 filas


## Validación cruzada: umbral y después limpieza

In [3]:
def kind(rule):
    return rule.split()[0]

out = []
for k in range(5):
    tr, te = folds.index[folds != k], folds.index[folds == k]
    base = R[R["limpieza"] == "ninguna"]
    m_tr = base[base.image.isin(tr)].groupby("regla")["matthews"].mean()
    picks = {"Otsu": "Otsu",
             "mejor fracción": m_tr[[r for r in m_tr.index if kind(r) == "fracción"]].idxmax(),
             "mejor histéresis": m_tr[[r for r in m_tr.index if kind(r) == "histéresis"]].idxmax()}
    for label in ("Otsu", "mejor histéresis"):
        rule = picks[label]
        sub = R[(R["regla"] == rule) & R.image.isin(tr)]
        g = sub.groupby("limpieza")[["matthews", "finos"]].mean()
        ok = g[g["finos"] >= g.loc["ninguna", "finos"] - 0.02]
        picks[label + " + limpieza"] = (rule, ok["matthews"].idxmax())
    for label, pick in picks.items():
        rule, clean = pick if isinstance(pick, tuple) else (pick, "ninguna")
        sel = R[(R["regla"] == rule) & (R["limpieza"] == clean) & R.image.isin(te)]
        for _, r in sel.iterrows():
            out.append({"vuelta": k, "candidata": label, "regla": rule, "limpieza": clean,
                        "image": r["image"], "matthews": r["matthews"], "finos": r["finos"]})
CV = pd.DataFrame(out)
print("Elecciones por vuelta:")
print(CV.drop_duplicates(["vuelta", "candidata"])[["vuelta", "candidata", "regla", "limpieza"]].to_string(index=False))

Elecciones por vuelta:
 vuelta                   candidata                regla                    limpieza
      0                        Otsu                 Otsu                     ninguna
      0              mejor fracción        fracción 0.09                     ninguna
      0            mejor histéresis histéresis 0.08-0.10                     ninguna
      0             Otsu + limpieza                 Otsu alargamiento 8, área 0.0005
      0 mejor histéresis + limpieza histéresis 0.08-0.10 alargamiento 8, área 0.0005
      1                        Otsu                 Otsu                     ninguna
      1              mejor fracción        fracción 0.09                     ninguna
      1            mejor histéresis histéresis 0.04-0.10                     ninguna
      1             Otsu + limpieza                 Otsu alargamiento 8, área 0.0005
      1 mejor histéresis + limpieza histéresis 0.04-0.10 alargamiento 8, área 0.0005
      2                        Otsu       

In [4]:
ORDER = ["Otsu", "mejor fracción", "mejor histéresis", "Otsu + limpieza", "mejor histéresis + limpieza"]
rows = []
for measure in ("matthews", "finos"):
    w = CV.pivot(index="image", columns="candidata", values=measure)[ORDER]
    d = w.sub(w["Otsu"], axis=0)
    lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
    for i, c in enumerate(ORDER):
        rows.append({"medida": measure, "candidata": c, "media afuera": w[c].mean(),
                     "vs Otsu": d[c].mean(), "límite inferior": lo[i], "límite superior": hi[i]})
res = pd.DataFrame(rows).set_index(["medida", "candidata"])
res.round(4)

media afuera  vs Otsu  límite inferior  \
medida   candidata                                                             
matthews Otsu                               0.6887   0.0000           0.0000   
         mejor fracción                     0.7022   0.0135           0.0063   
         mejor histéresis                   0.6989   0.0102           0.0005   
         Otsu + limpieza                    0.6918   0.0031           0.0011   
         mejor histéresis + limpieza        0.7024   0.0137           0.0045   
finos    Otsu                               0.1199   0.0000           0.0000   
         mejor fracción                     0.2190   0.0991           0.0719   
         mejor histéresis                   0.2313   0.1114           0.0813   
         Otsu + limpieza                    0.1110  -0.0090          -0.0123   
         mejor histéresis + limpieza        0.2269   0.1070           0.0771   

                                      límite superior  
medida   candidata                                     
matthews Otsu                                  0.0000  
         mejor fracción                        0.0210  
         mejor histéresis                      0.0195  
         Otsu + limpieza                       0.0064  
         mejor histéresis + limpieza           0.0227  
finos    Otsu                                  0.0000  
         mejor fracción                        0.1253  
         mejor histéresis                      0.1405  
         Otsu + limpieza                      -0.0064  
         mejor histéresis + limpieza           0.1356

In [5]:
print("Por grupo (Matthews y vasos finos, personas de afuera):")
CV.join(dev["group"], on="image").pivot_table(index="candidata", columns="group", values=["matthews", "finos"]).loc[ORDER].round(3)

Por grupo (Matthews y vasos finos, personas de afuera):


finos               matthews              
group                           dr      g      h       dr      g      h
candidata                                                              
Otsu                         0.177  0.098  0.084    0.658  0.698  0.709
mejor fracción               0.242  0.241  0.174    0.660  0.705  0.742
mejor histéresis             0.245  0.257  0.192    0.655  0.696  0.745
Otsu + limpieza              0.163  0.093  0.076    0.667  0.699  0.710
mejor histéresis + limpieza  0.238  0.255  0.188    0.662  0.698  0.747

## Decisión (escrita después de correr, 31/10)

| Regla (detector de líneas, mitad de resolución) | Matthews | Vasos finos |
|---|---|---|
| Otsu | 0,689 | 12 % |
| Fracción fija (9 % en las 5 vueltas) | 0,702 | 22 % |
| Histéresis | 0,699 | 23 % |
| Otsu + limpieza por forma | 0,692 | 11 % |
| Histéresis + limpieza por forma | 0,702 | 23 % |

- Frente a Otsu, la fracción fija y la histéresis casi duplican los vasos finos encontrados, en los tres grupos.
- **Histéresis contra fracción fija** (la regla ya adoptada en `05b`): Matthews −0,003 (−0,007 a +0,000), vasos finos +0,012 (+0,0003 a +0,025).
- **Limpieza por forma:** sube Matthews 0,003 pero baja los vasos finos, así que no cumple la condición (b) y **no se adopta**.

**Se mantiene la fracción fija; la histéresis no se adopta.** Al pie de la letra cumple la regla fijada (no inferior y mejora los finos con un intervalo que no incluye el cero), pero por muy poco: 1,2 puntos de vasos finos con un límite inferior de +0,0003, a cambio de un segundo umbral y la conexión entre píxeles, y solo probada sobre el detector de líneas, no sobre la combinación adoptada. Es una decisión de criterio a favor de la simplicidad, declarada. Se informa como probada, con una mejora mínima en vasos finos.
